In [18]:
import polars as pl
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import date

In [3]:
import kagglehub

path = kagglehub.dataset_download("desolution01/messy-employee-dataset")

In [4]:
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]

In [5]:
df = pl.read_csv(os.path.join(path,csv_file))

In [6]:
df.columns

['Employee_ID',
 'First_Name',
 'Last_Name',
 'Age',
 'Department_Region',
 'Status',
 'Join_Date',
 'Salary',
 'Email',
 'Phone',
 'Performance_Score',
 'Remote_Work']

In [7]:
df.head(100)

Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
str,str,str,i64,str,str,str,str,str,i64,str,bool
"""EMP1000""","""Bob""","""Davis""",25,"""DevOps-California""","""Active""","""4/2/2021""","""59767.65""","""bob.davis@example.com""",-1651623197,"""Average""",true
"""EMP1001""","""Bob""","""Brown""",null,"""Finance-Texas""","""Active""","""7/10/2020""","""65304.66""","""bob.brown@example.com""",-1898471390,"""Excellent""",true
"""EMP1002""","""Alice""","""Jones""",null,"""Admin-Nevada""","""Pending""","""12/7/2023""","""88145.9""","""alice.jones@example.com""",-5596363211,"""Good""",true
"""EMP1003""","""Eva""","""Davis""",25,"""Admin-Nevada""","""Inactive""","""11/27/2021""","""69450.99""","""eva.davis@example.com""",-3476490784,"""Good""",true
"""EMP1004""","""Frank""","""Williams""",25,"""Cloud Tech-Florida""","""Active""","""1/5/2022""","""109324.61""","""frank.williams@example.com""",-1586734256,"""Poor""",false
…,…,…,…,…,…,…,…,…,…,…,…
"""EMP1095""","""Heidi""","""Miller""",25,"""Cloud Tech-California""","""Active""","""4/8/2022""","""98425.1""","""heidi.miller@example.com""",-4610339936,"""Average""",true
"""EMP1096""","""Frank""","""Johnson""",40,"""Sales-Illinois""","""Inactive""","""5/21/2022""","""119764.2""","""frank.johnson@example.com""",-8861405785,"""Average""",false
"""EMP1097""","""Frank""","""Brown""",40,"""HR-California""","""Inactive""","""4/17/2021""","""58445.91""","""frank.brown@example.com""",-1706304890,"""Excellent""",false


In [8]:
print(df.schema)
print(df.null_count())
print(df.glimpse())

Schema({'Employee_ID': String, 'First_Name': String, 'Last_Name': String, 'Age': Int64, 'Department_Region': String, 'Status': String, 'Join_Date': String, 'Salary': String, 'Email': String, 'Phone': Int64, 'Performance_Score': String, 'Remote_Work': Boolean})
shape: (1, 12)
┌─────────────┬────────────┬───────────┬─────┬───┬───────┬───────┬───────────────────┬─────────────┐
│ Employee_ID ┆ First_Name ┆ Last_Name ┆ Age ┆ … ┆ Email ┆ Phone ┆ Performance_Score ┆ Remote_Work │
│ ---         ┆ ---        ┆ ---       ┆ --- ┆   ┆ ---   ┆ ---   ┆ ---               ┆ ---         │
│ u32         ┆ u32        ┆ u32       ┆ u32 ┆   ┆ u32   ┆ u32   ┆ u32               ┆ u32         │
╞═════════════╪════════════╪═══════════╪═════╪═══╪═══════╪═══════╪═══════════════════╪═════════════╡
│ 0           ┆ 0          ┆ 0         ┆ 211 ┆ … ┆ 0     ┆ 0     ┆ 0                 ┆ 0           │
└─────────────┴────────────┴───────────┴─────┴───┴───────┴───────┴───────────────────┴─────────────┘
Rows: 1020
Column

In [26]:

def split_department_region(df: pl.DataFrame) -> pl.DataFrame:

    return (
        df.with_columns(
            pl.col("Department_Region").str.split("-").list.get(0).alias("Department"),
            pl.col("Department_Region").str.split("-").list.get(1).alias("Region"),
        )
        .drop("Department_Region")
    )


def parse_dates(df: pl.DataFrame, cols: list[str], fmt: str) -> pl.DataFrame:

    return df.with_columns([
        pl.col(c).str.to_date(fmt, strict=False).alias(c)
        for c in cols
    ])


def cast_numeric(df: pl.DataFrame, schema: dict[str, pl.DataType]) -> pl.DataFrame:


    return df.with_columns([
        pl.col(c).cast(t, strict=False)
        for c, t in schema.items()
    ])


def nullify_invalid(df: pl.DataFrame, col: str, condition) -> pl.DataFrame:

    return df.with_columns(
        pl.when(condition)
          .then(None)
          .otherwise(pl.col(col))
          .alias(col)
    )


def fill_nulls_median(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:

    return df.with_columns([
        pl.col(c).fill_null(pl.col(c).median().cast(pl.Int64))
        for c in cols
    ])


def cast_enum(df: pl.DataFrame, schema: dict[str, list[str]]) -> pl.DataFrame:

    return df.with_columns([
        pl.col(c).cast(pl.Enum(vals))
        for c, vals in schema.items()
    ])



def drop_empty_columns(df: pl.DataFrame) -> pl.DataFrame:
    return df[[c for c in df.columns if df[c].null_count() < len(df)]]


def deduplicate(df: pl.DataFrame, subset: list[str], keep: str = "first") -> pl.DataFrame:
    n_before = len(df)
    n_dupes = df.filter(pl.col(subset[0]).is_duplicated()).shape[0]

    if n_dupes == 0:
        print(f"no duplicates found on {subset} - 0 duplicate rows, 0 dropped.")
        print(f"Rows before: {n_before}. Rows after: {n_before}")
        return df

    df = df.unique(subset = subset, keep=keep)

    n_after = len(df)
    n_dropped = n_before - n_after

    print(f"Duplicated found on {subset}: {n_dupes} duplicated rows, {n_dropped} dropped.")
    print(f"Rows before {n_before}. Rows after {n_after}")

    return df

def validate_emails(df: pl.DataFrame, col: str = "Email") -> pl.DataFrame:
    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    n_before = df[col].null_count()


    #nulify any email that does not match the pattern

    df = df.with_columns(pl.when(pl.col(col).str.contains(email_pattern)).then(pl.col(col)).otherwise(None).alias(col))

    n_after = df[col].null_count()
    n_invalid = n_after - n_before

    if n_invalid == 0:
        print(f"Email Validation: 0 invalid emails found")
    else:
        print(f"Email validation: {n_invalid} invalid emails nullified")

    return df

def add_tenure(df: pl.DataFrame, join_date_col: str = "Join_Date") -> pl.DataFrame:
    today = pl.lit(date.today())

    return df.with_columns(((today - pl.col(join_date_col)).dt.total_days()/365).cast(pl.Int32).alias("Tenure_Years"))
    
    
def add_salary_band(df: pl.DataFrame, col: str= "Salary") -> pl.DataFrame:

    p33 = df[col].quantile(0.33)
    p66 = df[col].quantile(0.66)

    return df.with_columns(pl.when(pl.col(col) < p33).then(pl.lit("Low")).when(pl.col(col) < p66).then(pl.lit("Mid")).otherwise(pl.lit("High")).cast(pl.Categorical).alias("Salary Band"))

def run_pipeline(df: pl.DataFrame) -> pl.DataFrame:

    # Step 1 — split compound column into Department and Region
    df = split_department_region(df)

    # Step 2 — parse Join_Date from M/D/YYYY string format to pl.Date
    df = parse_dates(df, cols=["Join_Date"], fmt="%m/%d/%Y")

    # Step 3 — cast Salary from string to float
    df = cast_numeric(df, {"Salary": pl.Float64})

    # Step 4 — nullify Phone values that are negative (corrupted data)
    df = nullify_invalid(df, "Phone", pl.col("Phone") < 0)

    #Step 5 - drop empty columns
    df = drop_empty_columns(df)  # drops Phone since it's 100% null
    
    # Step 6 — fill Age nulls with median (211 nulls found in triage)
    df = fill_nulls_median(df, ["Age"])

    # Step 7 — enforce known category sets for Status and Performance_Score
    df = cast_enum(df, {
        "Status": ["Active", "Inactive", "Pending"],
        "Performance_Score": ["Poor", "Average", "Good", "Excellent"]
    })

    #step 8 ensure unique employee id's drop duplicate records
    df = deduplicate(df,subset=["Employee_ID"], keep="first")

    #step 9 vailidating the emails
    df = validate_emails(df, col="Email")

    #Step 10 get the number of years worked based on the date joined
    df = add_tenure(df, join_date_col="Join_Date")

    #Step 11 segment the income to low mid high
    df = add_salary_band(df, col="Salary")
    
    return df


# --- Run the pipeline ---
df_clean = run_pipeline(df)

# Verify results
print(df_clean.schema)       # confirm if all dtypes are correct
print(df_clean.null_count()) # checking for null count

no duplicates found on ['Employee_ID'] - 0 duplicate rows, 0 dropped.
Rows before: 1020. Rows after: 1020
Email Validation: 0 invalid emails found
Schema({'Employee_ID': String, 'First_Name': String, 'Last_Name': String, 'Age': Int64, 'Status': Enum(categories=['Active', 'Inactive', 'Pending']), 'Join_Date': Date, 'Salary': Float64, 'Email': String, 'Performance_Score': Enum(categories=['Poor', 'Average', 'Good', 'Excellent']), 'Remote_Work': Boolean, 'Department': String, 'Region': String, 'Tenure_Years': Int32, 'Salary Band': Categorical})
shape: (1, 14)
┌─────────────┬────────────┬───────────┬─────┬───┬────────────┬────────┬─────────────┬─────────────┐
│ Employee_ID ┆ First_Name ┆ Last_Name ┆ Age ┆ … ┆ Department ┆ Region ┆ Tenure_Year ┆ Salary Band │
│ ---         ┆ ---        ┆ ---       ┆ --- ┆   ┆ ---        ┆ ---    ┆ s           ┆ ---         │
│ u32         ┆ u32        ┆ u32       ┆ u32 ┆   ┆ u32        ┆ u32    ┆ ---         ┆ u32         │
│             ┆            ┆     

In [24]:
df_clean.head()

Employee_ID,First_Name,Last_Name,Age,Status,Join_Date,Salary,Email,Performance_Score,Remote_Work,Department,Region,Tenure_Years,Salary Band
str,str,str,i64,enum,date,f64,str,enum,bool,str,str,i32,cat
"""EMP1000""","""Bob""","""Davis""",25,"""Active""",2021-04-02,59767.65,"""bob.davis@example.com""","""Average""",true,"""DevOps""","""California""",5,"""low"""
"""EMP1001""","""Bob""","""Brown""",30,"""Active""",2020-07-10,65304.66,"""bob.brown@example.com""","""Excellent""",true,"""Finance""","""Texas""",5,"""low"""
"""EMP1002""","""Alice""","""Jones""",30,"""Pending""",2023-12-07,88145.9,"""alice.jones@example.com""","""Good""",true,"""Admin""","""Nevada""",2,"""medium"""
"""EMP1003""","""Eva""","""Davis""",25,"""Inactive""",2021-11-27,69450.99,"""eva.davis@example.com""","""Good""",true,"""Admin""","""Nevada""",4,"""low"""
"""EMP1004""","""Frank""","""Williams""",25,"""Active""",2022-01-05,109324.61,"""frank.williams@example.com""","""Poor""",false,"""Cloud Tech""","""Florida""",4,"""high"""


In [27]:
df_clean.group_by("Salary Band").len().sort("Salary Band")

Salary Band,len
cat,u32
"""High""",363
"""Low""",328
"""Mid""",329
